# 🔷 Logika Fuzzy — Perancangan Rule Base & Inferensi Fuzzy
**Mata Kuliah:** Kecerdasan Buatan / Sistem Cerdas  
**Semester:** 4 — Teknik Elektro  

---

## 🎯 Tujuan Pembelajaran
Setelah menyelesaikan notebook ini, mahasiswa mampu:
1. Memahami konsep himpunan fuzzy dan fungsi keanggotaan
2. Merancang **Rule Base** (basis aturan) dalam sistem fuzzy
3. Melakukan proses **Fuzzifikasi**
4. Menerapkan **Inferensi Fuzzy** (Mamdani & Sugeno)
5. Melakukan **Defuzzifikasi** dan menginterpretasikan hasilnya

---

## 📌 Studi Kasus
### Sistem Kontrol Kecepatan Kipas Pendingin Otomatis
> Sebuah sistem kontrol ingin mengatur **kecepatan kipas** berdasarkan dua input:
> - **Suhu ruangan** (°C)
> - **Kelembaban** (%RH)
>
> Output: **Kecepatan Kipas** (%)

Ini adalah contoh nyata yang sering ditemui di sistem HVAC, data center, dan IoT.

---
## 📦 BAGIAN 0 — Instalasi & Import Library

In [ ]:
# Install library scikit-fuzzy jika belum ada
!pip install scikit-fuzzy -q

import numpy as np
import skfuzzy as fuzz
from skfuzzy import control as ctrl
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = '#f8f9fa'
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.4
plt.rcParams['font.size'] = 11

print('✅ Library berhasil diimport!')
print('   numpy      :', np.__version__)
print('   matplotlib : OK')
print('   scikit-fuzzy: OK')

---
## 📚 BAGIAN 1 — Konsep Dasar Himpunan Fuzzy

### 1.1 Apa itu Logika Fuzzy?

Logika klasik (Boolean) hanya mengenal **BENAR (1)** atau **SALAH (0)**.

Logika Fuzzy memungkinkan nilai **antara 0 dan 1** — mengikuti cara manusia berpikir.

Contoh: "Suhu 30°C — apakah PANAS?"
- Logika Boolean: Panas = Ya (1) / Tidak (0)
- Logika Fuzzy: μ_panas(30°C) = 0.7 → *agak panas*

### 1.2 Fungsi Keanggotaan (Membership Function)
Fungsi yang memetakan nilai crisp → derajat keanggotaan [0, 1]

In [ ]:
# ============================================================
# DEMO: Jenis-Jenis Fungsi Keanggotaan
# ============================================================
x = np.linspace(0, 100, 500)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Jenis-Jenis Fungsi Keanggotaan (Membership Function)', 
             fontsize=14, fontweight='bold', y=1.02)

# 1. Triangular (Segitiga)
mf_tri = fuzz.trimf(x, [20, 50, 80])
axes[0].plot(x, mf_tri, 'b-', linewidth=2.5)
axes[0].fill_between(x, mf_tri, alpha=0.15, color='blue')
axes[0].set_title('Triangular (Segitiga)\ntrimf([20, 50, 80])', fontweight='bold')
axes[0].set_xlabel('Nilai Input')
axes[0].set_ylabel('Derajat Keanggotaan μ')
axes[0].set_ylim(-0.05, 1.1)
axes[0].axvline(50, color='red', linestyle='--', alpha=0.6, label='Puncak = 50')
axes[0].legend()

# 2. Trapezoidal (Trapesium)
mf_trap = fuzz.trapmf(x, [15, 35, 65, 85])
axes[1].plot(x, mf_trap, 'g-', linewidth=2.5)
axes[1].fill_between(x, mf_trap, alpha=0.15, color='green')
axes[1].set_title('Trapezoidal (Trapesium)\ntrapmf([15, 35, 65, 85])', fontweight='bold')
axes[1].set_xlabel('Nilai Input')
axes[1].set_ylabel('Derajat Keanggotaan μ')
axes[1].set_ylim(-0.05, 1.1)

# 3. Gaussian
mf_gauss = fuzz.gaussmf(x, 50, 12)
axes[2].plot(x, mf_gauss, 'r-', linewidth=2.5)
axes[2].fill_between(x, mf_gauss, alpha=0.15, color='red')
axes[2].set_title('Gaussian\ngaussmf(mean=50, sigma=12)', fontweight='bold')
axes[2].set_xlabel('Nilai Input')
axes[2].set_ylabel('Derajat Keanggotaan μ')
axes[2].set_ylim(-0.05, 1.1)

plt.tight_layout()
plt.savefig('mf_types.png', dpi=120, bbox_inches='tight')
plt.show()
print('💡 Fungsi keanggotaan memetakan nilai crisp ke derajat [0, 1]')

---
## 🔧 BAGIAN 2 — Mendefinisikan Variabel Fuzzy & Fungsi Keanggotaan

### Langkah-langkah:
1. Tentukan **variabel** (input & output) beserta rentang nilainya
2. Tentukan **himpunan fuzzy** untuk setiap variabel (misal: RENDAH, SEDANG, TINGGI)
3. Definisikan **fungsi keanggotaan** untuk setiap himpunan

In [ ]:
# ============================================================
# DEFINISI VARIABEL FUZZY
# ============================================================

# --- INPUT 1: Suhu Ruangan (°C) ---
# Rentang: 0 - 45°C
suhu = ctrl.Antecedent(np.arange(0, 46, 1), 'suhu')

# --- INPUT 2: Kelembaban (%RH) ---
# Rentang: 0 - 100%
kelembaban = ctrl.Antecedent(np.arange(0, 101, 1), 'kelembaban')

# --- OUTPUT: Kecepatan Kipas (%) ---
# Rentang: 0 - 100%
kecepatan = ctrl.Consequent(np.arange(0, 101, 1), 'kecepatan_kipas')

print('📌 Variabel Fuzzy yang Didefinisikan:')
print('   INPUT  1 : Suhu Ruangan     → range [0, 45] °C')
print('   INPUT  2 : Kelembaban       → range [0, 100] %RH')
print('   OUTPUT   : Kecepatan Kipas  → range [0, 100] %')
print()

# ============================================================
# DEFINISI FUNGSI KEANGGOTAAN — SUHU
# ============================================================
suhu['dingin']  = fuzz.trapmf(suhu.universe,  [0,  0, 15, 22])
suhu['sejuk']   = fuzz.trimf(suhu.universe,   [18, 24, 30])
suhu['hangat']  = fuzz.trimf(suhu.universe,   [26, 32, 37])
suhu['panas']   = fuzz.trapmf(suhu.universe,  [34, 40, 45, 45])

# ============================================================
# DEFINISI FUNGSI KEANGGOTAAN — KELEMBABAN
# ============================================================
kelembaban['kering']   = fuzz.trapmf(kelembaban.universe, [0,  0,  25, 40])
kelembaban['normal']   = fuzz.trimf(kelembaban.universe,  [30, 50, 70])
kelembaban['lembab']   = fuzz.trimf(kelembaban.universe,  [60, 75, 88])
kelembaban['sangat_lembab'] = fuzz.trapmf(kelembaban.universe, [82, 92, 100, 100])

# ============================================================
# DEFINISI FUNGSI KEANGGOTAAN — KECEPATAN KIPAS
# ============================================================
kecepatan['mati']     = fuzz.trapmf(kecepatan.universe, [0,  0,  5,  15])
kecepatan['lambat']   = fuzz.trimf(kecepatan.universe,  [10, 25, 40])
kecepatan['sedang']   = fuzz.trimf(kecepatan.universe,  [35, 50, 65])
kecepatan['cepat']    = fuzz.trimf(kecepatan.universe,  [60, 75, 90])
kecepatan['max']      = fuzz.trapmf(kecepatan.universe, [85, 95, 100, 100])

print('✅ Fungsi keanggotaan berhasil didefinisikan!')
print('   Suhu       : dingin | sejuk | hangat | panas')
print('   Kelembaban : kering | normal | lembab | sangat_lembab')
print('   Kecepatan  : mati | lambat | sedang | cepat | max')

In [ ]:
# ============================================================
# VISUALISASI FUNGSI KEANGGOTAAN SEMUA VARIABEL
# ============================================================
fig, axes = plt.subplots(3, 1, figsize=(12, 12))
fig.suptitle('Fungsi Keanggotaan Variabel Fuzzy\nSistem Kontrol Kecepatan Kipas', 
             fontsize=14, fontweight='bold', y=0.98)

colors_suhu = ['#4A90D9', '#27AE60', '#E67E22', '#E74C3C']
labels_suhu = ['Dingin', 'Sejuk', 'Hangat', 'Panas']
mf_suhu = [suhu['dingin'].mf, suhu['sejuk'].mf, suhu['hangat'].mf, suhu['panas'].mf]

# Plot Suhu
ax = axes[0]
for mf, color, label in zip(mf_suhu, colors_suhu, labels_suhu):
    ax.plot(suhu.universe, mf, linewidth=2.5, color=color, label=label)
    ax.fill_between(suhu.universe, mf, alpha=0.1, color=color)
ax.set_title('INPUT 1: Suhu Ruangan (°C)', fontweight='bold', fontsize=12)
ax.set_xlabel('Suhu (°C)')
ax.set_ylabel('Derajat Keanggotaan μ')
ax.set_ylim(-0.05, 1.15)
ax.legend(loc='upper center', ncol=4, framealpha=0.9)

# Plot Kelembaban
colors_kel = ['#F39C12', '#2ECC71', '#3498DB', '#8E44AD']
labels_kel = ['Kering', 'Normal', 'Lembab', 'Sangat Lembab']
mf_kel = [kelembaban['kering'].mf, kelembaban['normal'].mf, 
          kelembaban['lembab'].mf, kelembaban['sangat_lembab'].mf]

ax = axes[1]
for mf, color, label in zip(mf_kel, colors_kel, labels_kel):
    ax.plot(kelembaban.universe, mf, linewidth=2.5, color=color, label=label)
    ax.fill_between(kelembaban.universe, mf, alpha=0.1, color=color)
ax.set_title('INPUT 2: Kelembaban (%RH)', fontweight='bold', fontsize=12)
ax.set_xlabel('Kelembaban (%RH)')
ax.set_ylabel('Derajat Keanggotaan μ')
ax.set_ylim(-0.05, 1.15)
ax.legend(loc='upper center', ncol=4, framealpha=0.9)

# Plot Kecepatan Kipas
colors_kec = ['#95A5A6', '#3498DB', '#27AE60', '#E67E22', '#E74C3C']
labels_kec = ['Mati', 'Lambat', 'Sedang', 'Cepat', 'Maksimum']
mf_kec = [kecepatan['mati'].mf, kecepatan['lambat'].mf, kecepatan['sedang'].mf,
          kecepatan['cepat'].mf, kecepatan['max'].mf]

ax = axes[2]
for mf, color, label in zip(mf_kec, colors_kec, labels_kec):
    ax.plot(kecepatan.universe, mf, linewidth=2.5, color=color, label=label)
    ax.fill_between(kecepatan.universe, mf, alpha=0.1, color=color)
ax.set_title('OUTPUT: Kecepatan Kipas (%)', fontweight='bold', fontsize=12)
ax.set_xlabel('Kecepatan Kipas (%)')
ax.set_ylabel('Derajat Keanggotaan μ')
ax.set_ylim(-0.05, 1.15)
ax.legend(loc='upper center', ncol=5, framealpha=0.9)

plt.tight_layout()
plt.savefig('membership_functions.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 📋 BAGIAN 3 — Perancangan Rule Base (Basis Aturan)

### Apa itu Rule Base?
Rule base adalah **sekumpulan aturan IF-THEN** yang merepresentasikan pengetahuan pakar.

Format umum:
```
IF (kondisi_input1) AND/OR (kondisi_input2) THEN (output)
```

### Strategi Perancangan Rule Base:
1. **Konsultasi pakar** (domain expert)
2. **Tabel keputusan** — buat matriks kombinasi semua input
3. **Data historis** — pelajari pola dari data

### Tabel Rule Base — Sistem Kipas Pendingin

| Suhu \ Kelembaban | Kering | Normal | Lembab | Sangat Lembab |
|:------------------:|:------:|:------:|:------:|:-------------:|
| **Dingin**         | Mati   | Lambat | Lambat | Sedang        |
| **Sejuk**          | Lambat | Lambat | Sedang | Cepat         |
| **Hangat**         | Sedang | Cepat  | Cepat  | Maksimum      |
| **Panas**          | Cepat  | Cepat  | Maksimum | Maksimum    |

In [ ]:
# ============================================================
# DEFINISI RULE BASE — 16 ATURAN
# ============================================================

# Format: IF suhu IS X AND kelembaban IS Y THEN kecepatan IS Z

rules = [
    # ---- Suhu DINGIN ----
    ctrl.Rule(suhu['dingin'] & kelembaban['kering'],       kecepatan['mati'],   label='R01'),
    ctrl.Rule(suhu['dingin'] & kelembaban['normal'],       kecepatan['lambat'], label='R02'),
    ctrl.Rule(suhu['dingin'] & kelembaban['lembab'],       kecepatan['lambat'], label='R03'),
    ctrl.Rule(suhu['dingin'] & kelembaban['sangat_lembab'],kecepatan['sedang'], label='R04'),
    
    # ---- Suhu SEJUK ----
    ctrl.Rule(suhu['sejuk']  & kelembaban['kering'],       kecepatan['lambat'], label='R05'),
    ctrl.Rule(suhu['sejuk']  & kelembaban['normal'],       kecepatan['lambat'], label='R06'),
    ctrl.Rule(suhu['sejuk']  & kelembaban['lembab'],       kecepatan['sedang'], label='R07'),
    ctrl.Rule(suhu['sejuk']  & kelembaban['sangat_lembab'],kecepatan['cepat'],  label='R08'),
    
    # ---- Suhu HANGAT ----
    ctrl.Rule(suhu['hangat'] & kelembaban['kering'],       kecepatan['sedang'], label='R09'),
    ctrl.Rule(suhu['hangat'] & kelembaban['normal'],       kecepatan['cepat'],  label='R10'),
    ctrl.Rule(suhu['hangat'] & kelembaban['lembab'],       kecepatan['cepat'],  label='R11'),
    ctrl.Rule(suhu['hangat'] & kelembaban['sangat_lembab'],kecepatan['max'],    label='R12'),
    
    # ---- Suhu PANAS ----
    ctrl.Rule(suhu['panas']  & kelembaban['kering'],       kecepatan['cepat'],  label='R13'),
    ctrl.Rule(suhu['panas']  & kelembaban['normal'],       kecepatan['cepat'],  label='R14'),
    ctrl.Rule(suhu['panas']  & kelembaban['lembab'],       kecepatan['max'],    label='R15'),
    ctrl.Rule(suhu['panas']  & kelembaban['sangat_lembab'],kecepatan['max'],    label='R16'),
]

print('=' * 55)
print('   RULE BASE — SISTEM KONTROL KIPAS PENDINGIN')
print('=' * 55)
rule_list = [
    ('R01', 'Dingin', 'Kering',       'Mati'),
    ('R02', 'Dingin', 'Normal',       'Lambat'),
    ('R03', 'Dingin', 'Lembab',       'Lambat'),
    ('R04', 'Dingin', 'Sangat Lembab','Sedang'),
    ('R05', 'Sejuk',  'Kering',       'Lambat'),
    ('R06', 'Sejuk',  'Normal',       'Lambat'),
    ('R07', 'Sejuk',  'Lembab',       'Sedang'),
    ('R08', 'Sejuk',  'Sangat Lembab','Cepat'),
    ('R09', 'Hangat', 'Kering',       'Sedang'),
    ('R10', 'Hangat', 'Normal',       'Cepat'),
    ('R11', 'Hangat', 'Lembab',       'Cepat'),
    ('R12', 'Hangat', 'Sangat Lembab','Maksimum'),
    ('R13', 'Panas',  'Kering',       'Cepat'),
    ('R14', 'Panas',  'Normal',       'Cepat'),
    ('R15', 'Panas',  'Lembab',       'Maksimum'),
    ('R16', 'Panas',  'Sangat Lembab','Maksimum'),
]
for no, s, k, out in rule_list:
    print(f'  [{no}] IF Suhu={s:<12} AND Kelembaban={k:<15} THEN Kipas={out}')
print('=' * 55)
print(f'  Total: {len(rules)} aturan')

---
## ⚙️ BAGIAN 4 — Proses Inferensi Fuzzy (Metode Mamdani)

### Alur Kerja Sistem Fuzzy:
```
Input Crisp → [FUZZIFIKASI] → Derajat Keanggotaan
                   ↓
             [INFERENSI] ← Rule Base
                   ↓
             [AGREGASI]
                   ↓
          [DEFUZZIFIKASI] → Output Crisp
```

### Metode Mamdani:
- **Fuzzifikasi**: Hitung μ setiap input terhadap semua MF
- **Evaluasi Aturan**: Gunakan operator MIN (AND) atau MAX (OR)
- **Agregasi**: Gabungkan semua output rule dengan MAX
- **Defuzzifikasi**: Centroid of Area (COA) / Center of Gravity

In [ ]:
# ============================================================
# MEMBANGUN SISTEM KONTROL FUZZY
# ============================================================

# Buat sistem kontrol dari rule base
sistem_kipas = ctrl.ControlSystem(rules)

# Buat simulator
simulator = ctrl.ControlSystemSimulation(sistem_kipas)

print('✅ Sistem Kontrol Fuzzy berhasil dibangun!')
print(f'   Metode defuzzifikasi: Centroid (default Mamdani)')
print()

# ============================================================
# CONTOH KOMPUTASI — LANGKAH DEMI LANGKAH
# ============================================================
SUHU_INPUT = 32       # °C — ganti nilai ini untuk eksperimen!
KELEMBABAN_INPUT = 70  # %RH — ganti nilai ini untuk eksperimen!

print('=' * 50)
print('  PERHITUNGAN INFERENSI FUZZY')
print('=' * 50)
print(f'  Input Crisp:')
print(f'    Suhu       = {SUHU_INPUT} °C')
print(f'    Kelembaban = {KELEMBABAN_INPUT} %RH')
print()

# ---- LANGKAH 1: FUZZIFIKASI ----
print('  LANGKAH 1 — FUZZIFIKASI')
print('  (Hitung derajat keanggotaan setiap input)')
print()

# Hitung μ Suhu
mu_dingin = fuzz.interp_membership(suhu.universe, suhu['dingin'].mf, SUHU_INPUT)
mu_sejuk  = fuzz.interp_membership(suhu.universe, suhu['sejuk'].mf, SUHU_INPUT)
mu_hangat = fuzz.interp_membership(suhu.universe, suhu['hangat'].mf, SUHU_INPUT)
mu_panas  = fuzz.interp_membership(suhu.universe, suhu['panas'].mf, SUHU_INPUT)

print(f'  μ_suhu(dingin)  = {mu_dingin:.4f}')
print(f'  μ_suhu(sejuk)   = {mu_sejuk:.4f}')
print(f'  μ_suhu(hangat)  = {mu_hangat:.4f}  ← dominan')
print(f'  μ_suhu(panas)   = {mu_panas:.4f}')
print()

# Hitung μ Kelembaban
mu_kering       = fuzz.interp_membership(kelembaban.universe, kelembaban['kering'].mf, KELEMBABAN_INPUT)
mu_normal       = fuzz.interp_membership(kelembaban.universe, kelembaban['normal'].mf, KELEMBABAN_INPUT)
mu_lembab       = fuzz.interp_membership(kelembaban.universe, kelembaban['lembab'].mf, KELEMBABAN_INPUT)
mu_sangat_lembab= fuzz.interp_membership(kelembaban.universe, kelembaban['sangat_lembab'].mf, KELEMBABAN_INPUT)

print(f'  μ_kelembaban(kering)       = {mu_kering:.4f}')
print(f'  μ_kelembaban(normal)       = {mu_normal:.4f}')
print(f'  μ_kelembaban(lembab)       = {mu_lembab:.4f}  ← dominan')
print(f'  μ_kelembaban(sangat_lembab)= {mu_sangat_lembab:.4f}')

In [ ]:
# ---- LANGKAH 2: EVALUASI ATURAN (INFERENSI) ----
print('=' * 55)
print('  LANGKAH 2 — EVALUASI ATURAN (INFERENSI)')
print('  Operator AND → MIN(μ_input1, μ_input2)')
print('=' * 55)
print()

# Hitung kekuatan setiap aturan (firing strength)
aturan_aktif = []

firing_data = [
    ('R01', mu_dingin, mu_kering,        'mati'),
    ('R02', mu_dingin, mu_normal,        'lambat'),
    ('R03', mu_dingin, mu_lembab,        'lambat'),
    ('R04', mu_dingin, mu_sangat_lembab, 'sedang'),
    ('R05', mu_sejuk,  mu_kering,        'lambat'),
    ('R06', mu_sejuk,  mu_normal,        'lambat'),
    ('R07', mu_sejuk,  mu_lembab,        'sedang'),
    ('R08', mu_sejuk,  mu_sangat_lembab, 'cepat'),
    ('R09', mu_hangat, mu_kering,        'sedang'),
    ('R10', mu_hangat, mu_normal,        'cepat'),
    ('R11', mu_hangat, mu_lembab,        'cepat'),
    ('R12', mu_hangat, mu_sangat_lembab, 'max'),
    ('R13', mu_panas,  mu_kering,        'cepat'),
    ('R14', mu_panas,  mu_normal,        'cepat'),
    ('R15', mu_panas,  mu_lembab,        'max'),
    ('R16', mu_panas,  mu_sangat_lembab, 'max'),
]

for no, mu1, mu2, output in firing_data:
    alpha = min(mu1, mu2)  # Operator AND = MIN
    status = '⚡ AKTIF' if alpha > 0 else '   -'
    if alpha > 0:
        aturan_aktif.append((no, alpha, output))
    print(f'  [{no}] MIN({mu1:.3f}, {mu2:.3f}) = {alpha:.4f}  {status}')

print()
print(f'  Aturan yang aktif: {len(aturan_aktif)} dari {len(firing_data)} aturan')

In [ ]:
# ---- LANGKAH 3: JALANKAN SIMULATOR ----
simulator.input['suhu'] = SUHU_INPUT
simulator.input['kelembaban'] = KELEMBABAN_INPUT
simulator.compute()

output_crisp = simulator.output['kecepatan_kipas']

print('=' * 50)
print('  LANGKAH 3 — DEFUZZIFIKASI (Centroid/COA)')
print('=' * 50)
print()
print(f'  Input  → Suhu: {SUHU_INPUT}°C, Kelembaban: {KELEMBABAN_INPUT}%RH')
print(f'  Output → Kecepatan Kipas: {output_crisp:.2f}%')
print()

# Interpretasi output
if output_crisp < 10:
    interp = '🔵 Kipas MATI — kondisi sudah nyaman'
elif output_crisp < 35:
    interp = '🟢 Kipas LAMBAT — kondisi cukup nyaman'
elif output_crisp < 60:
    interp = '🟡 Kipas SEDANG — perlu pendinginan'
elif output_crisp < 85:
    interp = '🟠 Kipas CEPAT — kondisi kurang nyaman'
else:
    interp = '🔴 Kipas MAKSIMUM — kondisi tidak nyaman!'

print(f'  Interpretasi: {interp}')

In [ ]:
# ============================================================
# VISUALISASI PROSES INFERENSI
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle(f'Proses Inferensi Fuzzy\nInput: Suhu={SUHU_INPUT}°C, Kelembaban={KELEMBABAN_INPUT}%RH → Output: {output_crisp:.1f}%',
             fontsize=13, fontweight='bold')

# --- Suhu ---
ax = axes[0]
colors_s = ['#4A90D9','#27AE60','#E67E22','#E74C3C']
labels_s = ['Dingin','Sejuk','Hangat','Panas']
mfs_s = [suhu['dingin'].mf, suhu['sejuk'].mf, suhu['hangat'].mf, suhu['panas'].mf]
mus_s = [mu_dingin, mu_sejuk, mu_hangat, mu_panas]

for mf, color, label, mu in zip(mfs_s, colors_s, labels_s, mus_s):
    ax.plot(suhu.universe, mf, linewidth=2, color=color, label=label)
    ax.fill_between(suhu.universe, mf, alpha=0.08, color=color)

ax.axvline(SUHU_INPUT, color='black', linewidth=2.5, linestyle='--', label=f'Input={SUHU_INPUT}°C')
ax.axhline(mu_hangat, color='#E67E22', linewidth=1.5, linestyle=':', alpha=0.8)
ax.annotate(f'μ_hangat={mu_hangat:.3f}', xy=(SUHU_INPUT, mu_hangat),
            xytext=(SUHU_INPUT+3, mu_hangat+0.05),
            fontsize=9, color='#E67E22',
            arrowprops=dict(arrowstyle='->', color='#E67E22'))
ax.set_title('INPUT 1: Suhu Ruangan', fontweight='bold')
ax.set_xlabel('Suhu (°C)')
ax.set_ylabel('Derajat Keanggotaan μ')
ax.set_ylim(-0.05, 1.2)
ax.legend(fontsize=9)

# --- Kelembaban ---
ax = axes[1]
colors_k = ['#F39C12','#2ECC71','#3498DB','#8E44AD']
labels_k = ['Kering','Normal','Lembab','Sangat Lembab']
mfs_k = [kelembaban['kering'].mf, kelembaban['normal'].mf, kelembaban['lembab'].mf, kelembaban['sangat_lembab'].mf]
mus_k = [mu_kering, mu_normal, mu_lembab, mu_sangat_lembab]

for mf, color, label, mu in zip(mfs_k, colors_k, labels_k, mus_k):
    ax.plot(kelembaban.universe, mf, linewidth=2, color=color, label=label)
    ax.fill_between(kelembaban.universe, mf, alpha=0.08, color=color)

ax.axvline(KELEMBABAN_INPUT, color='black', linewidth=2.5, linestyle='--', label=f'Input={KELEMBABAN_INPUT}%')
ax.axhline(mu_lembab, color='#3498DB', linewidth=1.5, linestyle=':', alpha=0.8)
ax.annotate(f'μ_lembab={mu_lembab:.3f}', xy=(KELEMBABAN_INPUT, mu_lembab),
            xytext=(KELEMBABAN_INPUT+5, mu_lembab+0.05),
            fontsize=9, color='#3498DB',
            arrowprops=dict(arrowstyle='->', color='#3498DB'))
ax.set_title('INPUT 2: Kelembaban', fontweight='bold')
ax.set_xlabel('Kelembaban (%RH)')
ax.set_ylabel('Derajat Keanggotaan μ')
ax.set_ylim(-0.05, 1.2)
ax.legend(fontsize=9)

# --- Output dengan garis defuzzifikasi ---
ax = axes[2]
colors_kec = ['#95A5A6','#3498DB','#27AE60','#E67E22','#E74C3C']
labels_kec = ['Mati','Lambat','Sedang','Cepat','Maksimum']
mfs_kec = [kecepatan['mati'].mf, kecepatan['lambat'].mf, kecepatan['sedang'].mf,
           kecepatan['cepat'].mf, kecepatan['max'].mf]

for mf, color, label in zip(mfs_kec, colors_kec, labels_kec):
    ax.plot(kecepatan.universe, mf, linewidth=2, color=color, label=label)
    ax.fill_between(kecepatan.universe, mf, alpha=0.08, color=color)

ax.axvline(output_crisp, color='black', linewidth=3, linestyle='--',
           label=f'Output={output_crisp:.1f}%')
ax.annotate(f'Defuzzifikasi\n{output_crisp:.1f}%', 
            xy=(output_crisp, 0.5), xytext=(output_crisp+8, 0.7),
            fontsize=10, fontweight='bold',
            arrowprops=dict(arrowstyle='->', color='black', lw=1.5))
ax.set_title('OUTPUT: Kecepatan Kipas', fontweight='bold')
ax.set_xlabel('Kecepatan Kipas (%)')
ax.set_ylabel('Derajat Keanggotaan μ')
ax.set_ylim(-0.05, 1.2)
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('inferensi_result.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 🔬 BAGIAN 5 — Analisis Permukaan Kontrol (Control Surface)

Permukaan kontrol menunjukkan **respons sistem** untuk seluruh kombinasi input.
Ini sangat berguna untuk memverifikasi apakah rule base yang kita rancang sudah sesuai.

In [ ]:
# ============================================================
# CONTROL SURFACE — 3D VISUALIZATION
# ============================================================
from mpl_toolkits.mplot3d import Axes3D
from matplotlib import cm

print('⏳ Menghitung control surface... (ini bisa memakan waktu 30-60 detik)')

# Grid sampling
suhu_range = np.linspace(0, 45, 25)
kel_range  = np.linspace(0, 100, 25)

S, K = np.meshgrid(suhu_range, kel_range)
Z = np.zeros_like(S)

for i in range(S.shape[0]):
    for j in range(S.shape[1]):
        try:
            sim_temp = ctrl.ControlSystemSimulation(sistem_kipas)
            sim_temp.input['suhu'] = S[i, j]
            sim_temp.input['kelembaban'] = K[i, j]
            sim_temp.compute()
            Z[i, j] = sim_temp.output['kecepatan_kipas']
        except:
            Z[i, j] = 0

# Plot 3D
fig = plt.figure(figsize=(14, 6))

# 3D Surface
ax1 = fig.add_subplot(121, projection='3d')
surf = ax1.plot_surface(S, K, Z, cmap='RdYlGn_r', alpha=0.85, edgecolor='none')
ax1.set_xlabel('Suhu (°C)', fontsize=11, labelpad=8)
ax1.set_ylabel('Kelembaban (%)', fontsize=11, labelpad=8)
ax1.set_zlabel('Kecepatan Kipas (%)', fontsize=11)
ax1.set_title('Control Surface 3D', fontweight='bold', fontsize=12)
fig.colorbar(surf, ax=ax1, shrink=0.5, label='Kecepatan (%)')
ax1.scatter([SUHU_INPUT], [KELEMBABAN_INPUT], [output_crisp], 
            color='blue', s=100, zorder=5, label='Input saat ini')
ax1.legend()

# Contour Plot 2D
ax2 = fig.add_subplot(122)
contour = ax2.contourf(S, K, Z, levels=20, cmap='RdYlGn_r')
ax2.contour(S, K, Z, levels=10, colors='white', alpha=0.3, linewidths=0.5)
plt.colorbar(contour, ax=ax2, label='Kecepatan Kipas (%)')
ax2.scatter(SUHU_INPUT, KELEMBABAN_INPUT, color='blue', s=150, 
            zorder=5, marker='*', label=f'Input ({SUHU_INPUT}°C, {KELEMBABAN_INPUT}%)')
ax2.set_xlabel('Suhu (°C)', fontsize=11)
ax2.set_ylabel('Kelembaban (%RH)', fontsize=11)
ax2.set_title('Peta Kontrol 2D (Contour Plot)', fontweight='bold', fontsize=12)
ax2.legend()

plt.tight_layout()
plt.savefig('control_surface.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ Control surface selesai dibuat!')

---
## 🧪 BAGIAN 6 — Eksperimen Mandiri

### 💡 Silakan ubah nilai input di bawah ini dan amati hasilnya!

In [ ]:
# ============================================================
#  ✏️  GANTI NILAI INI UNTUK EKSPERIMEN!
# ============================================================

SUHU_UJI       = 28    # Masukkan suhu (0 - 45 °C)
KELEMBABAN_UJI = 85    # Masukkan kelembaban (0 - 100 %RH)

# ============================================================
# (Tidak perlu mengubah kode di bawah ini)
# ============================================================

sim_uji = ctrl.ControlSystemSimulation(sistem_kipas)
sim_uji.input['suhu'] = SUHU_UJI
sim_uji.input['kelembaban'] = KELEMBABAN_UJI
sim_uji.compute()
hasil = sim_uji.output['kecepatan_kipas']

# Hitung μ untuk tampilan
mus = {
    'Dingin' : fuzz.interp_membership(suhu.universe, suhu['dingin'].mf, SUHU_UJI),
    'Sejuk'  : fuzz.interp_membership(suhu.universe, suhu['sejuk'].mf, SUHU_UJI),
    'Hangat' : fuzz.interp_membership(suhu.universe, suhu['hangat'].mf, SUHU_UJI),
    'Panas'  : fuzz.interp_membership(suhu.universe, suhu['panas'].mf, SUHU_UJI),
}
muk = {
    'Kering'      : fuzz.interp_membership(kelembaban.universe, kelembaban['kering'].mf, KELEMBABAN_UJI),
    'Normal'      : fuzz.interp_membership(kelembaban.universe, kelembaban['normal'].mf, KELEMBABAN_UJI),
    'Lembab'      : fuzz.interp_membership(kelembaban.universe, kelembaban['lembab'].mf, KELEMBABAN_UJI),
    'Sangat Lembab': fuzz.interp_membership(kelembaban.universe, kelembaban['sangat_lembab'].mf, KELEMBABAN_UJI),
}

print('╔══════════════════════════════════════════════╗')
print('║        HASIL INFERENSI FUZZY — EKSPERIMEN    ║')
print('╠══════════════════════════════════════════════╣')
print(f'║  Input Suhu       : {SUHU_UJI:5.1f} °C                   ║')
print(f'║  Input Kelembaban : {KELEMBABAN_UJI:5.1f} %RH                 ║')
print('╠══════════════════════════════════════════════╣')
print('║  Fuzzifikasi Suhu:')
for label, mu_val in mus.items():
    bar = '█' * int(mu_val * 20)
    print(f'║    {label:<8}: {mu_val:.4f} |{bar:<20}|')
print('║  Fuzzifikasi Kelembaban:')
for label, mu_val in muk.items():
    bar = '█' * int(mu_val * 20)
    print(f'║    {label:<14}: {mu_val:.4f} |{bar:<20}|')
print('╠══════════════════════════════════════════════╣')
print(f'║  ➤ OUTPUT Kecepatan Kipas: {hasil:6.2f} %           ║')
print('╚══════════════════════════════════════════════╝')

---
## 📊 BAGIAN 7 — Perbandingan Berbagai Skenario

In [ ]:
# ============================================================
# UJI BERBAGAI SKENARIO SEKALIGUS
# ============================================================

skenario = [
    ('Kondisi Dingin & Kering',    10, 20),
    ('Kondisi Sejuk & Normal',     24, 50),
    ('Kondisi Hangat & Lembab',    32, 70),
    ('Kondisi Panas & Sangat Lembab', 40, 90),
    ('Kondisi Ekstrem Panas',      45, 95),
    ('Kondisi Nyaman',             22, 45),
]

print('┌─────────────────────────────────┬────────┬────────┬────────────┐')
print('│ Skenario                        │ Suhu   │ Kelbt  │ Kec.Kipas  │')
print('├─────────────────────────────────┼────────┼────────┼────────────┤')

hasil_skenario = []
for nama, s, k in skenario:
    sim_s = ctrl.ControlSystemSimulation(sistem_kipas)
    sim_s.input['suhu'] = s
    sim_s.input['kelembaban'] = k
    sim_s.compute()
    out = sim_s.output['kecepatan_kipas']
    hasil_skenario.append(out)
    print(f'│ {nama:<31} │ {s:4}°C  │ {k:4}%   │ {out:7.2f} %   │')

print('└─────────────────────────────────┴────────┴────────┴────────────┘')

# Visualisasi perbandingan
fig, ax = plt.subplots(figsize=(12, 5))
nama_skenario = [s[0] for s in skenario]
colors_bar = ['#3498DB' if v < 33 else '#27AE60' if v < 60 else '#E67E22' if v < 80 else '#E74C3C'
              for v in hasil_skenario]

bars = ax.barh(nama_skenario, hasil_skenario, color=colors_bar, edgecolor='white', height=0.6)
ax.set_xlabel('Kecepatan Kipas (%)', fontsize=12)
ax.set_title('Perbandingan Output Fuzzy untuk Berbagai Skenario', fontweight='bold', fontsize=13)
ax.set_xlim(0, 110)

for bar, val in zip(bars, hasil_skenario):
    ax.text(val + 1, bar.get_y() + bar.get_height()/2, 
            f'{val:.1f}%', va='center', fontweight='bold', fontsize=11)

legend_elements = [
    mpatches.Patch(facecolor='#3498DB', label='Mati/Lambat (< 33%)'),
    mpatches.Patch(facecolor='#27AE60', label='Sedang (33-60%)'),
    mpatches.Patch(facecolor='#E67E22', label='Cepat (60-80%)'),
    mpatches.Patch(facecolor='#E74C3C', label='Maksimum (> 80%)'),
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=10)
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('skenario_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 🆚 BAGIAN 8 — Metode Defuzzifikasi (Perbandingan)

Tiga metode defuzzifikasi yang umum:
| Metode | Singkatan | Keterangan |
|--------|-----------|------------|
| **Centroid of Area** | COA | Titik pusat massa area — paling umum |
| **Mean of Maximum** | MOM | Rata-rata nilai dengan μ tertinggi |
| **Bisector** | BIS | Garis yang membagi area menjadi dua sama besar |

In [ ]:
# ============================================================
# PERBANDINGAN METODE DEFUZZIFIKASI
# ============================================================

# Buat simulasi dengan metode berbeda
metode_list = ['centroid', 'mom', 'bisector']
nama_metode  = ['Centroid (COA)', 'Mean of Maximum (MOM)', 'Bisector']

SUHU_DEF = 30
KELEMBABAN_DEF = 65

print(f'Input: Suhu={SUHU_DEF}°C, Kelembaban={KELEMBABAN_DEF}%RH')
print()
print('┌──────────────────────────┬───────────────┐')
print('│ Metode Defuzzifikasi     │ Output (%)    │')
print('├──────────────────────────┼───────────────┤')

hasil_defuzz = []
for metode, nama in zip(metode_list, nama_metode):
    kec_tmp = ctrl.Consequent(np.arange(0, 101, 1), 'kecepatan_kipas', defuzzify_method=metode)
    kec_tmp['mati']   = fuzz.trapmf(kec_tmp.universe, [0,  0,  5,  15])
    kec_tmp['lambat'] = fuzz.trimf(kec_tmp.universe,  [10, 25, 40])
    kec_tmp['sedang'] = fuzz.trimf(kec_tmp.universe,  [35, 50, 65])
    kec_tmp['cepat']  = fuzz.trimf(kec_tmp.universe,  [60, 75, 90])
    kec_tmp['max']    = fuzz.trapmf(kec_tmp.universe, [85, 95, 100, 100])
    
    rules_tmp = [
        ctrl.Rule(suhu['dingin'] & kelembaban['kering'],       kec_tmp['mati']),
        ctrl.Rule(suhu['dingin'] & kelembaban['normal'],       kec_tmp['lambat']),
        ctrl.Rule(suhu['dingin'] & kelembaban['lembab'],       kec_tmp['lambat']),
        ctrl.Rule(suhu['dingin'] & kelembaban['sangat_lembab'],kec_tmp['sedang']),
        ctrl.Rule(suhu['sejuk']  & kelembaban['kering'],       kec_tmp['lambat']),
        ctrl.Rule(suhu['sejuk']  & kelembaban['normal'],       kec_tmp['lambat']),
        ctrl.Rule(suhu['sejuk']  & kelembaban['lembab'],       kec_tmp['sedang']),
        ctrl.Rule(suhu['sejuk']  & kelembaban['sangat_lembab'],kec_tmp['cepat']),
        ctrl.Rule(suhu['hangat'] & kelembaban['kering'],       kec_tmp['sedang']),
        ctrl.Rule(suhu['hangat'] & kelembaban['normal'],       kec_tmp['cepat']),
        ctrl.Rule(suhu['hangat'] & kelembaban['lembab'],       kec_tmp['cepat']),
        ctrl.Rule(suhu['hangat'] & kelembaban['sangat_lembab'],kec_tmp['max']),
        ctrl.Rule(suhu['panas']  & kelembaban['kering'],       kec_tmp['cepat']),
        ctrl.Rule(suhu['panas']  & kelembaban['normal'],       kec_tmp['cepat']),
        ctrl.Rule(suhu['panas']  & kelembaban['lembab'],       kec_tmp['max']),
        ctrl.Rule(suhu['panas']  & kelembaban['sangat_lembab'],kec_tmp['max']),
    ]
    try:
        sys_tmp = ctrl.ControlSystem(rules_tmp)
        sim_tmp = ctrl.ControlSystemSimulation(sys_tmp)
        sim_tmp.input['suhu'] = SUHU_DEF
        sim_tmp.input['kelembaban'] = KELEMBABAN_DEF
        sim_tmp.compute()
        out_tmp = sim_tmp.output['kecepatan_kipas']
    except:
        out_tmp = 0
    hasil_defuzz.append(out_tmp)
    print(f'│ {nama:<24} │ {out_tmp:10.2f} % │')

print('└──────────────────────────┴───────────────┘')
print()
print('📝 Catatan: COA (Centroid) paling sering digunakan karena smooth dan stabil.')

---
## 📝 BAGIAN 9 — Latihan Soal

### Soal 1
Tambahkan aturan baru ke rule base:
- *"Jika suhu PANAS dan kelembaban KERING, maka kecepatan SEDANG"*  
  (alasannya: meski panas, jika kering, pendinginan alami masih cukup efektif)

Ubah Rule R13 di Bagian 3, lalu jalankan ulang sistem dan amati perbedaannya.

### Soal 2
Desain sistem fuzzy baru dengan kasus:
> **Kontrol Intensitas Lampu Belajar**
> - Input: Tingkat Cahaya Ruangan (lux), Waktu (jam)
> - Output: Intensitas Lampu (%)

Tentukan:
1. Himpunan fuzzy setiap variabel
2. Fungsi keanggotaan yang sesuai
3. Minimal 9 rule
4. Implementasikan dan uji dengan beberapa skenario

### Soal 3 (Analisis)
Perhatikan control surface yang sudah dibuat. Mengapa ada daerah di mana output berubah secara gradual? Jelaskan kaitannya dengan konsep overlap pada fungsi keanggotaan!

In [ ]:
# ============================================================
# TEMPLATE KOSONG UNTUK LATIHAN SOAL 2
# ============================================================

# TODO: Definisikan variabel fuzzy untuk sistem lampu belajar
# cahaya = ctrl.Antecedent(np.arange(??, ??, ?), 'cahaya')
# waktu  = ctrl.Antecedent(np.arange(??, ??, ?), 'waktu')
# lampu  = ctrl.Consequent(np.arange(??, ??, ?), 'intensitas_lampu')

# TODO: Definisikan membership function
# cahaya['gelap']  = fuzz.??mf(cahaya.universe, [...])
# cahaya['redup']  = fuzz.??mf(cahaya.universe, [...])
# ...

# TODO: Buat rule base (minimal 9 aturan)
# rules_lampu = [
#     ctrl.Rule(cahaya['gelap'] & waktu['malam'], lampu['terang']),
#     ...
# ]

print('💡 Silakan isi template di atas untuk mengerjakan Soal 2!')
print('   Gunakan kode di Bagian 2-4 sebagai referensi.')

---
## 📚 Ringkasan

| Tahap | Proses | Teknik |
|-------|--------|--------|
| 1 | **Definisi Variabel** | Tentukan input, output, dan rentang nilai |
| 2 | **Fungsi Keanggotaan** | Triangular, Trapezoidal, Gaussian |
| 3 | **Rule Base** | IF-THEN rules dari pengetahuan pakar |
| 4 | **Fuzzifikasi** | Hitung μ setiap input |
| 5 | **Inferensi** | MIN (AND), MAX (OR), evaluasi aturan |
| 6 | **Agregasi** | Gabungkan output semua aturan |
| 7 | **Defuzzifikasi** | COA, MOM, atau Bisector → nilai crisp |

---
## 📖 Referensi
- Zadeh, L.A. (1965). *Fuzzy Sets*. Information and Control, 8(3), 338-353.
- Mamdani, E.H. & Assilian, S. (1975). *An experiment in linguistic synthesis with a fuzzy logic controller*.
- Klir, G.J. & Yuan, B. (1995). *Fuzzy Sets and Fuzzy Logic: Theory and Applications*. Prentice Hall.
- scikit-fuzzy documentation: https://pythonhosted.org/scikit-fuzzy/

---
*Dibuat untuk mata kuliah Kecerdasan Buatan — Teknik Elektro*